# 01 - Dataset 基础

## 学习目标

1. 理解为什么需要 Dataset
2. 掌握 Dataset 的三个核心方法
3. 学会使用最简单的 Dataset
4. 理解延迟加载的设计思想

## 1. 为什么需要 Dataset？

在深度学习中，我们经常要处理大量数据。如果直接将所有数据加载到内存，会遇到以下问题：

- **内存溢出**：数据集太大，内存放不下
- **训练效率低**：无法利用 GPU 的并行计算能力
- **代码复杂**：需要手动处理批量加载、数据打乱、多进程等

PyTorch 的 Dataset 提供了优雅的解决方案：**延迟加载**

```
初始化时：只记录数据路径（不占内存）
使用时：才真正加载数据（按需加载）
```

## 2. Dataset 的三个核心方法

| 方法 | 作用 | 返回值 |
|------|------|--------|
| `__init__()` | 初始化数据集，记录数据路径 | 无 |
| `__len__()` | 返回数据集大小 | int |
| `__getitem__(index)` | 根据索引获取单个样本 | (data, label) |

**类比理解：**

把 Dataset 想象成图书馆的目录系统：
- `__init__`：建立目录索引（记录每本书在哪个书架）
- `__len__`：告诉你图书馆有多少本书
- `__getitem__`：根据编号去书架上取书

## 3. 最简单的 Dataset 示例

In [ ]:
import torch
from torch.utils.data import Dataset

class SimpleDataset(Dataset):
    """最简单的 Dataset 示例"""
    
    def __init__(self, data_list):
        """
        初始化数据集
        data_list: 数据列表，例如 [(x1, y1), (x2, y2), ...]
        """
        self.data = data_list
        print(f"数据集初始化完成，共有 {len(data_list)} 个样本")
    
    def __getitem__(self, index):
        """返回第 index 个样本"""
        print(f"正在获取第 {index} 个样本...")
        return self.data[index]
    
    def __len__(self):
        """返回数据集大小"""
        return len(self.data)

# 创建数据：5 个 (输入, 输出) 对
data = [(1, 10), (2, 20), (3, 30), (4, 40), (5, 50)]

# 实例化 Dataset
dataset = SimpleDataset(data)

print(f"\n数据集大小: {len(dataset)}")
print(f"第 0 个样本: {dataset[0]}")
print(f"第 2 个样本: {dataset[2]}")

**关键观察：**

1. `dataset[0]` 会自动调用 `__getitem__(0)`
2. `len(dataset)` 会自动调用 `__len__()`
3. 可以像操作列表一样操作 Dataset

## 4. 遍历 Dataset

In [ ]:
# 方式 1：使用索引遍历
print("=== 方式 1：使用索引 ===")
for i in range(len(dataset)):
    x, y = dataset[i]
    print(f"样本 {i}: x={x}, y={y}")

In [ ]:
# 方式 2：直接遍历（更 Pythonic）
print("\n=== 方式 2：直接遍历 ===")
for i, (x, y) in enumerate(dataset):
    print(f"样本 {i}: x={x}, y={y}")

## 5. 数学数据集示例

In [ ]:
import torch

class LinearDataset(Dataset):
    """线性数据集：y = 2*x + 1"""
    
    def __init__(self, num_samples):
        """
        num_samples: 要生成的样本数量
        """
        self.num_samples = num_samples
        # 注意：这里只记录数量，不生成数据
    
    def __getitem__(self, index):
        """动态生成第 index 个样本"""
        x = torch.tensor([float(index)], dtype=torch.float32)
        y = 2 * x + 1  # y = 2x + 1
        return x, y
    
    def __len__(self):
        return self.num_samples

# 创建数据集
dataset = LinearDataset(num_samples=10)

print(f"数据集大小: {len(dataset)}\n")

# 查看前 5 个样本
for i in range(5):
    x, y = dataset[i]
    print(f"样本 {i}: x={x.item():.1f}, y={y.item():.1f}")

## 6. 理解延迟加载

In [ ]:
import time

class LazyDataset(Dataset):
    """演示延迟加载的 Dataset"""
    
    def __init__(self, n):
        print("=== 初始化 Dataset ===")
        self.n = n
        print(f"只记录数量 n={n}，不加载数据")
        print("初始化完成（很快！）\n")
    
    def __getitem__(self, index):
        print(f"现在才加载第 {index} 个数据...")
        time.sleep(0.1)  # 模拟从硬盘读取数据的耗时
        data = index * index  # 模拟数据处理
        print(f"加载完成: {data}\n")
        return data
    
    def __len__(self):
        return self.n

# 创建 1000 个样本的数据集
print("开始创建数据集...")
start = time.time()
dataset = LazyDataset(1000)
end = time.time()
print(f"创建耗时: {end-start:.4f} 秒\n")

# 只访问 3 个样本
print("只访问 3 个样本：")
for i in [0, 10, 100]:
    data = dataset[i]

**关键理解：**

1. **初始化很快**：只记录元信息，不加载数据
2. **按需加载**：只在访问时才加载数据
3. **节省内存**：不需要的数据不会占用内存

这就是为什么 Dataset 能处理百万级数据集的原因！

## 7. 练习题

### 练习 1：创建平方数据集

创建一个数据集，返回 (x, x²) 对

In [ ]:
class SquareDataset(Dataset):
    """返回 (x, x²) 的数据集"""
    
    def __init__(self, n):
        # TODO: 实现初始化
        pass
    
    def __getitem__(self, index):
        # TODO: 返回 (x, x²)
        pass
    
    def __len__(self):
        # TODO: 返回数据集大小
        pass

# 测试代码
# dataset = SquareDataset(10)
# for i in range(5):
#     x, y = dataset[i]
#     print(f"x={x}, x²={y}")

### 练习 2：字符串数据集

创建一个数据集，存储字符串及其长度

In [ ]:
class StringDataset(Dataset):
    """字符串数据集"""
    
    def __init__(self, string_list):
        # TODO: 保存字符串列表
        pass
    
    def __getitem__(self, index):
        # TODO: 返回 (字符串, 长度)
        pass
    
    def __len__(self):
        # TODO: 返回列表长度
        pass

# 测试代码
# strings = ["hello", "world", "pytorch", "dataset"]
# dataset = StringDataset(strings)
# for i in range(len(dataset)):
#     text, length = dataset[i]
#     print(f"文本: {text}, 长度: {length}")

## 8. 小结

### 核心要点

1. **Dataset 的作用**：提供统一的数据访问接口
2. **三个核心方法**：`__init__`, `__len__`, `__getitem__`
3. **延迟加载**：初始化时不加载数据，使用时才加载
4. **像列表一样使用**：支持索引访问和遍历

### 设计思想

```python
# Dataset 的设计模式
__init__:    记录元信息（路径、大小等）  → 快速
__getitem__: 按需加载数据                → 节省内存
__len__:     返回数据集大小              → 方便迭代
```

### 下一步

在下一个 notebook 中，我们将学习如何创建**自定义 Dataset**，处理真实的图像数据！

---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！